# 02 Clean: population and roof aggregates

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/buildings_fr.parquet`, `data/processed/roofs_fr.parquet`.
Outputs: `data/processed/buildings_clean.parquet` (one row per building of the population, with roof aggregates; no heating columns) and `data/processed/heating_clean.parquet` (heating columns of the same buildings, kept apart for the sensitivity run only).

Population: existing (`gstat` 1004) residential (`gkat` 1020, 1030, 1040) buildings.

In [ ]:
import helpers
import pandas as pd

Load the two extracts written by stage 01.

In [ ]:
buildings = pd.read_parquet(helpers.PROCESSED / "buildings_fr.parquet")
roofs = pd.read_parquet(helpers.PROCESSED / "roofs_fr.parquet")
print(len(buildings), "buildings,", len(roofs), "roof surfaces")

Type the columns we need (the register is read as strings). Missing values become NaN.

In [ ]:
numeric = [
    "egid",
    "ggdenr",
    "gkode",
    "gkodn",
    "gstat",
    "gkat",
    "gklas",
    "gbauj",
    "gbaup",
    "garea",
    "gastw",
    "ganzwhg",
]
for column in numeric:
    buildings[column] = pd.to_numeric(buildings[column])
buildings["egid"] = buildings["egid"].astype("int64")
print(buildings["egid"].is_unique)

Population: existing residential buildings.

In [ ]:
population = buildings[
    (buildings["gstat"] == 1004) & buildings["gkat"].isin([1020, 1030, 1040])
].copy()
print("population:", len(population))

Roof aggregates per building. Suitable surfaces are those with class 3 or better.

In [ ]:
suitable = roofs[roofs["klasse"] >= helpers.SUITABLE_CLASS_MIN]
all_surfaces = roofs.groupby("egid").agg(
    n_surfaces=("objectid", "count"),
    roof_area_m2=("flaeche", "sum"),
    best_class=("klasse", "max"),
)
good_surfaces = suitable.groupby("egid").agg(
    suitable_area_m2=("flaeche", "sum"),
    yield_kwh=("stromertrag", "sum"),
)
# A building with roof records but no suitable surface has 0 suitable area and 0 yield
aggregates = all_surfaces.join(good_surfaces)
aggregates[["suitable_area_m2", "yield_kwh"]] = aggregates[
    ["suitable_area_m2", "yield_kwh"]
].fillna(0)
aggregates = aggregates.reset_index()

Left-join onto the population. Buildings without any roof record keep missing aggregates.

In [ ]:
clean = population.merge(aggregates, on="egid", how="left")
keep = [
    "egid",
    "ggdenr",
    "ggdename",
    "gkode",
    "gkodn",
    "gkat",
    "gklas",
    "gbauj",
    "gbaup",
    "garea",
    "gastw",
    "ganzwhg",
    "n_surfaces",
    "roof_area_m2",
    "suitable_area_m2",
    "yield_kwh",
    "best_class",
]
clean = clean[keep]
print("population size:", len(clean))
print("share without any roof record:", round(clean["n_surfaces"].isna().mean(), 4))
small = clean["suitable_area_m2"] < helpers.MIN_SUITABLE_ROOF_M2
print("share with suitable area < 20 m2 (roof record, small):", round(small.mean(), 4))
print(
    "share without usable roof (no record or < 20 m2):",
    round((small | clean["suitable_area_m2"].isna()).mean(), 4),
)

In [ ]:
assert clean["egid"].is_unique
clean.to_parquet(helpers.PROCESSED / "buildings_clean.parquet", index=False)

## Heating columns (sensitivity run only)

The main model never sees heating (date leakage, docs/decisions.md). The three columns go to their own file, one row per population building, in the order of `buildings_clean`: `genh1` (energy source of the main heating, register code), `gwaerzh1` (heat generator code) and `gwaerdath1` (date the heating record was last updated). Codes are kept as the register's integers; their labels are in the register's code list (`kodes_codes_codici.csv` in `fr.zip`) and are grouped in stage 03.

In [ ]:
# Same population and row order as buildings_clean; codes as nullable integers, date as datetime
heating = population[["egid", "genh1", "gwaerzh1", "gwaerdath1"]].copy()
for column in ["genh1", "gwaerzh1"]:
    heating[column] = pd.to_numeric(heating[column]).astype("Int64")
heating["gwaerdath1"] = pd.to_datetime(heating["gwaerdath1"])
heating = heating.reset_index(drop=True)
assert heating["egid"].is_unique
assert (heating["egid"].to_numpy() == clean["egid"].to_numpy()).all()
share_missing = heating[["genh1", "gwaerzh1", "gwaerdath1"]].isna().mean()
print("missing share:", share_missing.round(3).to_dict())
heating.to_parquet(helpers.PROCESSED / "heating_clean.parquet", index=False)